In [1]:
"""
Statistical Benchmarking — Wilcoxon Signed-Rank Test

Project: Forecasting Weekly Dengue Cases in Mindanao Cities (Bandiola & Balani)

Benchmarks the models across 10 random seeds:
  - Proposed: xgb_full (Full XGBoost — 28 features incl. weather)
  - Baseline: xgb_case_only (ablation: 6 case/seasonality features)
  - Reference: persistence (naive baseline)

One-tailed Wilcoxon signed-rank test (α = 0.05).
Comparison tables: mean ± std, W, p-value, % gain.
Runtime benchmarking + per-city tests + plots.

Models trained: 2016–2019 | Test: 2020–2021 (70/30 chronological split)
"""

import sys, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding="utf-8", errors="replace")
    except (AttributeError, ValueError):
        pass

PROJECT = Path(r"C:\Users\princ\miniconda3\envs\forecasting")
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

from config import XGB_PARAMS, ALL_FEATURES, CASE_ONLY_FEATURES
from src.features.build_features import build_feature_matrix

print("Project:", PROJECT)
print("Full features:", len(ALL_FEATURES), "| Case-only:", len(CASE_ONLY_FEATURES))

Project: C:\Users\princ\miniconda3\envs\forecasting
Full features: 28 | Case-only: 6


In [2]:
# Load data & build feature matrix

In [3]:
try:
    from src.data_loader import load_real_panel
    df = load_real_panel()
except Exception as exc:
    print(f"Real panel unavailable ({exc}); generating synthetic data.")
    from data.generate_synthetic_data import generate_panel, add_derived_weather
    df = add_derived_weather(generate_panel())

X, y, city, splits, dates = build_feature_matrix(df)
print(f"X={X.shape}  train={splits['train'].sum()}  test={splits['test'].sum()}")

Loading real panels from: C:\Users\princ\miniconda3\envs\forecasting\datasets
  read dengue_davao_city_week_panel_2016_2021.csv: 262 rows
  read dengue_cagayan_de_oro_city_week_panel_2016_2021.csv: 262 rows
  read dengue_zamboanga_city_week_panel_2016_2021.csv: 262 rows
  merged: 786 rows, 37 columns
  Dropping 3 row(s) with week_number > 52
  Collapsing 6 duplicate city-year-week rows (median of reported values)
  Dropping 9 row(s) with missing weekly_dengue_cases
  Cagayan de Oro City: winsorising 2 week(s) above cap 484 (max 998)
  Zamboanga City: winsorising 1 week(s) above cap 513 (max 3382)
  final: 771 rows, 3 cities, years 2016.0-2020.0
Feature matrix: 753 rows x 28 features
  Features: 28 (expected 28)
  Train:      606 rows  (2016–2019)
  Embargo:    3 rows  (skipped)
  Test (2020–2021): 144 rows
X=(753, 28)  train=606  test=144


In [4]:
# Benchmark runner: 3 models × 10 seeds

In [5]:
N_RUNS = 10
SEEDS = list(range(42, 42 + N_RUNS))
MODEL_NAMES = ["persistence", "xgb_case_only", "xgb_full"]
BASELINES = ["persistence", "xgb_case_only"]

test_idx = np.where(splits["test"])[0]
y_test = y[splits["test"]].values
city_test = city.values[test_idx]


def _metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


from src.models.models import ResidualXGBRegressor
from config import XGB_PARAMS, SHRINKAGE, ALL_FEATURES, CASE_ONLY_FEATURES


def train_one(name, seed):
    t0 = time.time()
    if name == "persistence":
        pred_test = X.loc[splits["test"], "cases_lag1"].values.astype(float)
    else:
        if name == "xgb_case_only":
            feats = CASE_ONLY_FEATURES
            params = {**XGB_PARAMS, "random_state": seed}
        elif name == "xgb_full":
            feats = ALL_FEATURES
            params = {**XGB_PARAMS, "random_state": seed}
        else:
            raise ValueError(name)
        mdl = ResidualXGBRegressor(feature_set=feats, shrinkage=SHRINKAGE, **params)
        mdl.fit(X.loc[splits["train"]], y[splits["train"]])
        pred_test = mdl.predict(X.loc[splits["test"]])
    runtime = time.time() - t0
    return {
        "seed": seed, "runtime_sec": runtime,
        "test": _metrics(y_test, pred_test),
        "pred_test": pred_test,
    }


print(f"Running {len(MODEL_NAMES)} models x {N_RUNS} seeds ...")
raw_results = {m: [] for m in MODEL_NAMES}
for m in MODEL_NAMES:
    for s in SEEDS:
        raw_results[m].append(train_one(m, s))
    print(f"  done: {m}")


Running 3 models x 10 seeds ...
  done: persistence
  done: xgb_case_only
  done: xgb_full


In [6]:
# Tidy DataFrame + descriptive stats

In [7]:
rows = []
for m in MODEL_NAMES:
    for r in raw_results[m]:
        rows.append({
            "model": m, "seed": r["seed"], "runtime_sec": r["runtime_sec"],
            "test_MAE": r["test"]["MAE"], "test_RMSE": r["test"]["RMSE"],
            "test_R2": r["test"]["R2"],
        })
bench_df = pd.DataFrame(rows)

desc = bench_df.groupby("model").agg(
    MAE_mean=("test_MAE", "mean"), MAE_std=("test_MAE", "std"),
    RMSE_mean=("test_RMSE", "mean"), RMSE_std=("test_RMSE", "std"),
    R2_mean=("test_R2", "mean"), R2_std=("test_R2", "std"),
    time_mean=("runtime_sec", "mean"), time_std=("runtime_sec", "std"),
).round(4)
print("=== Mean ± Std across 10 seeds (2020–2021 Test Year) ===")
print(desc.to_string())

bench_df.to_csv("outputs/benchmark_runs.csv", index=False)
print("\nSaved: outputs/benchmark_runs.csv")


def extract(model, metric):
    col = "runtime_sec" if metric == "runtime_sec" else f"test_{metric}"
    return bench_df.loc[bench_df["model"] == model, col].values

=== Mean ± Std across 10 seeds (2020–2021 Test Year) ===
               MAE_mean  MAE_std  RMSE_mean  RMSE_std  R2_mean  R2_std  time_mean  time_std
model                                                                                      
persistence      6.0069    0.000     9.2508    0.0000   0.1879  0.0000     0.0003    0.0005
xgb_case_only    5.8696    0.017     8.9721    0.0206   0.2361  0.0035     0.0237    0.0302
xgb_full         5.8141    0.021     8.8959    0.0177   0.2490  0.0030     0.0178    0.0017

Saved: outputs/benchmark_runs.csv


In [8]:
# Wilcoxon comparison function

In [9]:
def compare_and_print_table(proposed, baseline, metric_name,
                            proposed_label="Full XGBoost",
                            baseline_label="Baseline",
                            higher_is_better=False, alpha=0.05):
    proposed = np.asarray(proposed, dtype=float)
    baseline = np.asarray(baseline, dtype=float)

    if higher_is_better:
        gain = proposed - baseline
        alternative = "greater"
    else:
        gain = baseline - proposed
        alternative = "less"

    df_cmp = pd.DataFrame({
        "Run": range(1, len(proposed) + 1),
        proposed_label: proposed.round(4),
        baseline_label: baseline.round(4),
        "Gain (better if +)": gain.round(4),
    })
    print(f"\n===== {metric_name} Comparison =====")
    print(f"Proposed : {proposed_label}   |   Baseline : {baseline_label}")
    print(df_cmp.to_string(index=False))

    try:
        stat, p_value = wilcoxon(proposed, baseline, alternative=alternative)
    except ValueError:
        stat, p_value = 0.0, 1.0

    direction = "Proposed > Baseline" if higher_is_better else "Proposed < Baseline"
    print(f"\nWilcoxon Signed-Rank Test (One-Tailed: {direction})")
    print(f"Statistic (W): {stat}")
    print(f"P-value      : {p_value:.4f}")
    if p_value < alpha:
        print(f"Result: Significant improvement at alpha={alpha}")
    else:
        print(f"Result: No statistically significant difference at alpha={alpha}")

    return {
        "metric": metric_name,
        "baseline_label": baseline_label,
        "W": float(stat), "p": float(p_value),
        "significant": bool(p_value < alpha),
        "mean_proposed": float(np.mean(proposed)),
        "std_proposed": float(np.std(proposed, ddof=1)),
        "mean_baseline": float(np.mean(baseline)),
        "std_baseline": float(np.std(baseline, ddof=1)),
        "gain_pct": float(np.mean(gain) / np.mean(baseline) * 100)
                    if np.mean(baseline) != 0 else float("nan"),
    }

In [10]:
# MAE benchmark (lower is better)

In [11]:
PROPOSED = "xgb_full"
mae_results = [
    compare_and_print_table(
        extract(PROPOSED, "MAE"), extract(b, "MAE"),
        metric_name="Test MAE (2020–2021)",
        proposed_label="Full XGBoost", baseline_label=b,
        higher_is_better=False,
    ) for b in BASELINES
]


===== Test MAE (2020–2021) Comparison =====
Proposed : Full XGBoost   |   Baseline : persistence
 Run  Full XGBoost  persistence  Gain (better if +)
   1        5.7668       6.0069              0.2402
   2        5.8321       6.0069              0.1748
   3        5.8211       6.0069              0.1858
   4        5.8202       6.0069              0.1867
   5        5.8085       6.0069              0.1984
   6        5.8052       6.0069              0.2017
   7        5.8236       6.0069              0.1834
   8        5.8316       6.0069              0.1753
   9        5.8360       6.0069              0.1710
  10        5.7955       6.0069              0.2114

Wilcoxon Signed-Rank Test (One-Tailed: Proposed < Baseline)
Statistic (W): 0.0
P-value      : 0.0010
Result: Significant improvement at alpha=0.05

===== Test MAE (2020–2021) Comparison =====
Proposed : Full XGBoost   |   Baseline : xgb_case_only
 Run  Full XGBoost  xgb_case_only  Gain (better if +)
   1        5.7668         5

In [12]:
# RMSE benchmark (lower is better)

In [13]:
rmse_results = [
    compare_and_print_table(
        extract(PROPOSED, "RMSE"), extract(b, "RMSE"),
        metric_name="Test RMSE (2020–2021)",
        proposed_label="Full XGBoost", baseline_label=b,
        higher_is_better=False,
    ) for b in BASELINES
]


===== Test RMSE (2020–2021) Comparison =====
Proposed : Full XGBoost   |   Baseline : persistence
 Run  Full XGBoost  persistence  Gain (better if +)
   1        8.8665       9.2508              0.3843
   2        8.9112       9.2508              0.3395
   3        8.9103       9.2508              0.3405
   4        8.8830       9.2508              0.3677
   5        8.9107       9.2508              0.3401
   6        8.9044       9.2508              0.3463
   7        8.9040       9.2508              0.3467
   8        8.8783       9.2508              0.3724
   9        8.9140       9.2508              0.3368
  10        8.8768       9.2508              0.3740

Wilcoxon Signed-Rank Test (One-Tailed: Proposed < Baseline)
Statistic (W): 0.0
P-value      : 0.0010
Result: Significant improvement at alpha=0.05

===== Test RMSE (2020–2021) Comparison =====
Proposed : Full XGBoost   |   Baseline : xgb_case_only
 Run  Full XGBoost  xgb_case_only  Gain (better if +)
   1        8.8665        

In [14]:
# R² benchmark (higher is better)

In [15]:
r2_results = [
    compare_and_print_table(
        extract(PROPOSED, "R2"), extract(b, "R2"),
        metric_name="Test R² (2020–2021)",
        proposed_label="Full XGBoost", baseline_label=b,
        higher_is_better=True,
    ) for b in BASELINES
]


===== Test R² (2020–2021) Comparison =====
Proposed : Full XGBoost   |   Baseline : persistence
 Run  Full XGBoost  persistence  Gain (better if +)
   1        0.2540       0.1879              0.0661
   2        0.2464       0.1879              0.0585
   3        0.2466       0.1879              0.0587
   4        0.2512       0.1879              0.0633
   5        0.2465       0.1879              0.0586
   6        0.2476       0.1879              0.0597
   7        0.2476       0.1879              0.0597
   8        0.2520       0.1879              0.0641
   9        0.2459       0.1879              0.0581
  10        0.2522       0.1879              0.0643

Wilcoxon Signed-Rank Test (One-Tailed: Proposed > Baseline)
Statistic (W): 55.0
P-value      : 0.0010
Result: Significant improvement at alpha=0.05

===== Test R² (2020–2021) Comparison =====
Proposed : Full XGBoost   |   Baseline : xgb_case_only
 Run  Full XGBoost  xgb_case_only  Gain (better if +)
   1        0.2540         0.

In [16]:
# Runtime benchmark (lower is better)

In [17]:
time_results = [
    compare_and_print_table(
        extract(PROPOSED, "runtime_sec"), extract(b, "runtime_sec"),
        metric_name="Training Runtime (seconds)",
        proposed_label="Full XGBoost", baseline_label=b,
        higher_is_better=False,
    ) for b in BASELINES
]


===== Training Runtime (seconds) Comparison =====
Proposed : Full XGBoost   |   Baseline : persistence
 Run  Full XGBoost  persistence  Gain (better if +)
   1        0.0170        0.001             -0.0160
   2        0.0160        0.000             -0.0160
   3        0.0207        0.000             -0.0207
   4        0.0160        0.001             -0.0150
   5        0.0196        0.000             -0.0196
   6        0.0184        0.000             -0.0184
   7        0.0166        0.000             -0.0166
   8        0.0158        0.000             -0.0158
   9        0.0187        0.001             -0.0177
  10        0.0191        0.000             -0.0191

Wilcoxon Signed-Rank Test (One-Tailed: Proposed < Baseline)
Statistic (W): 55.0
P-value      : 1.0000
Result: No statistically significant difference at alpha=0.05

===== Training Runtime (seconds) Comparison =====
Proposed : Full XGBoost   |   Baseline : xgb_case_only
 Run  Full XGBoost  xgb_case_only  Gain (better if +)

In [18]:
# Master summary table

In [19]:
def significance_stars(p):
    if p < 0.01:
        return "YES (p<0.01)"
    elif p < 0.05:
        return "YES (p<0.05)"
    return "no (n.s.)"


ALL_GROUPS = [
    ("Test MAE (2020–2021)", mae_results),
    ("Test RMSE (2020–2021)", rmse_results),
    ("Test R² (2020–2021)", r2_results),
    ("Runtime (s)", time_results),
]

rows = []
idx = 1
for group_name, group in ALL_GROUPS:
    for res in group:
        rows.append({
            "#": idx,
            "Metric": group_name,
            "Baseline Model": res["baseline_label"],
            "Baseline (mean ± std)": f"{res['mean_baseline']:.4f} ± {res['std_baseline']:.4f}",
            "Full XGB (mean ± std)": f"{res['mean_proposed']:.4f} ± {res['std_proposed']:.4f}",
            "Gain": f"{res['gain_pct']:+.2f}%",
            "W": res["W"],
            "p-Value": f"{res['p']:.4f}",
            "Significant?": significance_stars(res["p"]),
        })
        idx += 1

summary_tbl = pd.DataFrame(rows)
print("=" * 110)
print("TABLE: STATISTICAL BENCHMARK — Full XGBoost (Proposed) vs Baselines")
print("One-Tailed Wilcoxon Signed-Rank Test | alpha = 0.05 | 10 seeds (42-51)")
print("=" * 110)
print(summary_tbl.to_string(index=False))

summary_tbl.to_csv("outputs/wilcoxon_benchmark_summary.csv", index=False)
print("\nSaved: outputs/wilcoxon_benchmark_summary.csv")

TABLE: STATISTICAL BENCHMARK — Full XGBoost (Proposed) vs Baselines
One-Tailed Wilcoxon Signed-Rank Test | alpha = 0.05 | 10 seeds (42-51)
 #                Metric Baseline Model Baseline (mean ± std) Full XGB (mean ± std)      Gain    W p-Value Significant?
 1  Test MAE (2020–2021)    persistence       6.0069 ± 0.0000       5.8141 ± 0.0210    +3.21%  0.0  0.0010 YES (p<0.01)
 2  Test MAE (2020–2021)  xgb_case_only       5.8696 ± 0.0170       5.8141 ± 0.0210    +0.95%  0.0  0.0010 YES (p<0.01)
 3 Test RMSE (2020–2021)    persistence       9.2508 ± 0.0000       8.8959 ± 0.0177    +3.84%  0.0  0.0010 YES (p<0.01)
 4 Test RMSE (2020–2021)  xgb_case_only       8.9721 ± 0.0206       8.8959 ± 0.0177    +0.85%  0.0  0.0010 YES (p<0.01)
 5   Test R² (2020–2021)    persistence       0.1879 ± 0.0000       0.2490 ± 0.0030   +32.52% 55.0  0.0010 YES (p<0.01)
 6   Test R² (2020–2021)  xgb_case_only       0.2361 ± 0.0035       0.2490 ± 0.0030    +5.47% 55.0  0.0010 YES (p<0.01)
 7           Runtime 

In [20]:
# Per-city benchmark + Wilcoxon

In [21]:
cities_unique = sorted(set(city_test))
per_city_records = []
for m in ["persistence", "xgb_case_only", "xgb_full"]:
    for s in SEEDS:
        if m == "persistence":
            pred = X.loc[splits["test"], "cases_lag1"].values.astype(float)
        else:
            feats = CASE_ONLY_FEATURES if m == "xgb_case_only" else ALL_FEATURES
            params = {**XGB_PARAMS, "random_state": s}
            mdl = ResidualXGBRegressor(feature_set=feats, shrinkage=SHRINKAGE, **params)
            mdl.fit(X.loc[splits["train"]], y[splits["train"]])
            pred = mdl.predict(X.loc[splits["test"]])
        for c in cities_unique:
            cm = city_test == c
            per_city_records.append({
                "model": m, "seed": s, "city": c,
                "MAE": mean_absolute_error(y_test[cm], pred[cm]),
            })

per_city_df = pd.DataFrame(per_city_records)
print("=" * 80)
print("PER-CITY MAE across 10 seeds (2020-2021 Test)")
print("=" * 80)
print(per_city_df.groupby(["city", "model"])["MAE"].agg(["mean", "std"]).round(3).to_string())


PER-CITY MAE across 10 seeds (2020-2021 Test)
                                    mean    std
city                model                      
Cagayan de Oro City persistence    6.521  0.000
                    xgb_case_only  6.422  0.027
                    xgb_full       6.293  0.030
Davao City          persistence    7.854  0.000
                    xgb_case_only  7.510  0.035
                    xgb_full       7.386  0.047
Zamboanga City      persistence    3.646  0.000
                    xgb_case_only  3.676  0.041
                    xgb_full       3.763  0.039


In [22]:
# Visualizations

In [23]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#4e79a7", "#f28e2b", "#e15759"]

mae_data = [extract(m, "MAE") for m in MODEL_NAMES]
bp = axes[0].boxplot(mae_data, tick_labels=MODEL_NAMES, patch_artist=True)
for patch, col in zip(bp["boxes"], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
axes[0].set_title("Test MAE across 10 Seeds (2020–2021)")
axes[0].set_ylabel("MAE (cases)")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(True, alpha=0.3, axis="y")

time_data = [bench_df.loc[bench_df["model"] == m, "runtime_sec"].values for m in MODEL_NAMES]
bp2 = axes[1].boxplot(time_data, tick_labels=MODEL_NAMES, patch_artist=True)
for patch, col in zip(bp2["boxes"], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
axes[1].set_title("Training Runtime across 10 Seeds")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(True, alpha=0.3, axis="y")

pvals = [r["p"] for r in mae_results]
bars = axes[2].bar(BASELINES, pvals, color=colors[1:])
axes[2].axhline(0.05, color="red", linestyle="--", label="alpha = 0.05")
axes[2].set_title("Wilcoxon p-values — Full XGB vs Baselines (MAE)")
axes[2].set_ylabel("p-value")
axes[2].set_ylim(0, max(1.0, max(pvals) * 1.3))
axes[2].legend()
axes[2].tick_params(axis="x", rotation=15)
for bar, pv in zip(bars, pvals):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{pv:.4f}", ha="center", fontsize=9)
axes[2].grid(True, alpha=0.3, axis="y")

plt.suptitle("Statistical Benchmark — Full XGBoost vs Baselines (10 Seeds)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figures/wilcoxon_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/figures/wilcoxon_benchmark.png")

Saved: outputs/figures/wilcoxon_benchmark.png


In [24]:
# Auto-generated interpretation

In [25]:
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)
print(f"""
One-tailed Wilcoxon signed-rank tests (alpha = 0.05) were conducted across
{N_RUNS} independent training runs (seeds 42-51) comparing the proposed
Full XGBoost model against each baseline on the held-out 2020–2021 test year.

Key findings (Test MAE, 2020–2021):
""")
for res in mae_results:
    verdict = ("statistically significant improvement"
               if res["significant"] else "no statistically significant difference")
    print(f"  * Full XGB vs {res['baseline_label']:>14s}:  "
          f"p = {res['p']:.4f}  ->  {verdict}")

print(f"""
Gain (%) is the average relative improvement of Full XGBoost over the
baseline; positive values mean the proposed model is better. A significant
result (p < 0.05) indicates the observed difference is unlikely under the
null hypothesis of no difference.
""")

INTERPRETATION

One-tailed Wilcoxon signed-rank tests (alpha = 0.05) were conducted across
10 independent training runs (seeds 42-51) comparing the proposed
Full XGBoost model against each baseline on the held-out 2020–2021 test year.

Key findings (Test MAE, 2020–2021):

  * Full XGB vs    persistence:  p = 0.0010  ->  statistically significant improvement
  * Full XGB vs  xgb_case_only:  p = 0.0010  ->  statistically significant improvement

Gain (%) is the average relative improvement of Full XGBoost over the
baseline; positive values mean the proposed model is better. A significant
result (p < 0.05) indicates the observed difference is unlikely under the
null hypothesis of no difference.

